In [ ]:
import numpy as np
from datasets import load_dataset

# 뉴스 요약 데이터셋을 불러옴 (테스트 분할만 사용)
news = load_dataset("argilla/news-summary", split="test")

# 데이터를 pandas DataFrame으로 변환하고 5000개 샘플을 무작위로 선택 (재현성을 위해 random_state 고정)
df = news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]

# "text" 열의 각 텍스트 앞에 "summarize: "를 추가하여 모델에 요약 지시를 명시
df["text"] = "summarize: " + df["text"]

# "prediction" 열에서 요약 텍스트만 추출 (리스트 내 딕셔너리 구조에서 "text" 값만 가져옴)
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])

# DataFrame을 학습(60%), 검증(20%), 테스트(20%) 데이터로 분할
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

# 학습 데이터의 첫 번째 원본 뉴스와 요약을 예시로 출력 (원본은 200자, 요약은 50자로 제한)
print(f"Source News : {train.text.iloc[0][:200]}")
print(f"Summarization : {train.prediction.iloc[0][:50]}")

# 각 데이터셋의 크기를 출력
print(f"Training Data Size : {len(train)}")
print(f"Validation Data Size : {len(valid)}")
print(f"Testing Data Size : {len(test)}")

/home/teom142/.conda/envs/pytorch_study/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Source News : summarize: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, we
Summarization : Putin says had useful interaction with Trump at Vi
Training Data Size : 3000
Validation Data Size : 1000
Testing Data Size : 1000


In [ ]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

# 데이터셋을 토큰화하고 TensorDataset으로 변환하는 함수 정의
def make_dataset(data, tokenizer, device):
    # 원본 텍스트를 토큰화 (최대 길이 128, 패딩 및 잘림 적용)
    source = tokenizer(
        text=data.text.tolist(),
        padding="max_length",
        max_length=128,
        pad_to_max_length=True,
        truncation=True,
        return_tensors="pt"
    )

    # 목표 요약 텍스트를 토큰화 (최대 길이 128, 패딩 및 잘림 적용)
    target = tokenizer(
        text=data.prediction.tolist(),
        padding="max_length", 
        max_length=128,
        pad_to_max_length=True,
        truncation=True,
        return_tensors="pt"
    )

    # 토큰화된 데이터를 텐서로 추출하고 지정된 디바이스(GPU/CPU)로 이동
    source_ids = source["input_ids"].squeeze().to(device)
    source_mask = source["attention_mask"].squeeze().to(device)
    target_ids = target["input_ids"].squeeze().to(device)
    target_mask = target["attention_mask"].squeeze().to(device)

    # TensorDataset으로 반환 (입력 ID, 마스크, 목표 ID, 목표 마스크 포함)
    return TensorDataset(source_ids, source_mask, target_ids, target_mask)

# DataLoader를 생성하는 함수 정의
def get_datalodader(dataset, sampler, batch_size):

    # 데이터셋에 사용할 샘플러 생성 (랜덤 또는 순차적)
    data_sampler = sampler(dataset)

    # DataLoader 객체 생성 (배치 크기와 샘플러 적용)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

# 하이퍼파라미터 설정
epochs = 5  # 학습 에포크 수
batch_size = 8  # 배치 크기
device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU 사용 가능 여부에 따라 디바이스 선택

# T5 토크나이저 초기화 ("t5-small" 모델 사용)
tokenizer = T5Tokenizer.from_pretrained(
    pretrained_model_name_or_path="t5-small"
)

# 학습, 검증, 테스트 데이터셋 생성
train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_datalodader(train_dataset, RandomSampler, batch_size)  # 학습용: 랜덤 샘플링

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_datalodader(valid_dataset, SequentialSampler, batch_size)  # 검증용: 순차 샘플링

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_datalodader(test_dataset, SequentialSampler, batch_size)  # 테스트용: 순차 샘플링

# 학습 DataLoader의 첫 번째 배치를 출력 (데이터 구조 확인용)
print(next(iter(train_dataloader)))

# 특정 토큰 ID에 해당하는 토큰 출력 (특수 토큰 확인용)
print(tokenizer.convert_ids_to_tokens(21603))  # 예: </s> (문장 종료 토큰)
print(tokenizer.convert_ids_to_tokens(10))     # 예: 공백 또는 기타 특수 토큰

/home/teom142/.conda/envs/pytorch_study/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. If you see this, DO NOT PANIC! This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


[tensor([[21603,    10,  8161,  ...,  1190,     3,     1],
        [21603,    10,   549,  ...,  3193,  4683,     1],
        [21603,    10,    41,  ...,  1702,    16,     1],
        ...,
        [21603,    10,   549,  ...,  1747,    16,     1],
        [21603,    10,   549,  ...,    26,    12,     1],
        [21603,    10, 24686,  ...,   344,  4262,     1]], device='cuda:0'), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), tensor([[ 5199,   159, 22038,  ...,     0,     0,     0],
        [ 2523,  3602,  1131,  ...,     0,     0,     0],
        [11543,  2689,    10,  ...,     0,     0,     0],
        ...,
        [  412,     5,   134,  ...,     0,     0,     0],
        [  412,     5,   134,  ...,     0,     0,     0],
        [  262,   189,  2532,  ...,     0,     0,     0]], device='cuda:0'), ten

In [ ]:
from torch import optim
from transformers import T5ForConditionalGeneration

# T5 조건부 생성 모델 초기화 ("t5-small" 사용) 및 디바이스로 이동
model = T5ForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="t5-small",
).to(device)

# AdamW 옵티마이저 설정 (학습률 1e-5, epsilon 1e-8)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

In [5]:
import numpy as np
from torch import nn

# 정확도 계산 함수 (참고: 생성 모델에 적합하지 않음, 분류 모델용으로 보임)
def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

# 모델 학습 함수 정의
def train(model, optimizer, dataloader):
    model.train()  # 모델을 학습 모드로 설정
    train_loss = 0.0  # 학습 손실 초기화
    for source_ids, source_mask, target_ids, target_mask in dataloader:

        # 디코더 입력 ID를 목표 ID에서 한 칸 이동하여 준비
        decoder_input_ids = target_ids[:, :-1].contiguous()

        # 라벨 준비 (패딩 토큰은 -100으로 설정하여 손실 계산에서 제외)
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        # 모델에 데이터 전달하여 손실 계산
        outputs = model(
            input_ids=source_ids,
            attention_mask=source_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
        )
        loss = outputs.loss
        train_loss += loss.item()

        # 역전파 및 옵티마이저 스텝
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # 평균 학습 손실 계산
    train_loss = train_loss / len(dataloader)
    return train_loss

# 모델 평가 함수 정의
def evaluation(model, dataloader):
    with torch.no_grad():  # 그래디언트 계산 비활성화
        model.eval()  # 모델을 평가 모드로 설정
        val_loss = 0.0  # 검증 손실 초기화
        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()
            labels = target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100
            outputs = model(
                input_ids=source_ids,
                attention_mask=source_mask,
                decoder_input_ids=decoder_input_ids,
                labels=labels,
            )
            loss = outputs.loss
            val_loss += loss.item()

    # 평균 검증 손실 계산
    val_loss = val_loss / len(dataloader)
    return val_loss

# 학습 루프 실행
best_loss = 10000  # 최적 손실 초기값 설정
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)  # 학습
    val_loss = evaluation(model, valid_dataloader)  # 검증
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")  # 손실 출력
    
    # 검증 손실이 개선되면 모델 저장
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "../models/T5ForConditionalGeneration.pt")
        print("Saved the model weights")

Epoch 1: Train Loss: 3.8050 Val Loss: 3.1155
Saved the model weights
Epoch 2: Train Loss: 3.2875 Val Loss: 2.8494
Saved the model weights
Epoch 3: Train Loss: 3.0748 Val Loss: 2.7254
Saved the model weights
Epoch 4: Train Loss: 2.9490 Val Loss: 2.6486
Saved the model weights
Epoch 5: Train Loss: 2.8543 Val Loss: 2.5937
Saved the model weights


In [6]:
# 모델 테스트
model.eval()  # 모델을 평가 모드로 설정
with torch.no_grad():
    for source_ids, source_mask, target_ids, target_mask in test_dataloader:

        # 빔 서치를 사용하여 요약 생성
        generated_ids = model.generate(
            input_ids=source_ids,
            attention_mask=source_mask,
            max_length=128,
            num_beams=3,  # 빔 서치 빔 수
            repetition_penalty=2.5,  # 반복 페널티
            length_penalty=1.0,  # 길이 페널티
            early_stopping=True,  # 조기 종료 활성화
        )
        
        # 생성된 요약과 실제 요약 비교 출력
        for generated, target in zip(generated_ids, target_ids):
            pred = tokenizer.decode(
                generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            actual = tokenizer.decode(
                target, skip_special_tokens=True, clean_up_tokenization_spaces=True
            )
            print("Generated Headline Text:", pred)  # 생성된 헤드라인
            print("Actual Headline Text   :", actual)  # 실제 헤드라인
        break  # 첫 번째 배치만 처리 (데모용)

Generated Headline Text: Clinton leads Trump by 4 percentage points in four-war race for Nov. 8 election
Actual Headline Text   : Clinton leads Trump by 4 points in Washington Post: ABC News poll
Generated Headline Text: U.S. senators sharpen potential line of attack against Gorsuch's nomination to Supreme Court
Actual Headline Text   : Democrats question independence of Trump Supreme Court nominee
Generated Headline Text: U.S. warns Saudi Arabia over Yemen's humanitarian situation could constrain U.S. aid, a U.S. official says
Actual Headline Text   : In push for Yemen aid, U.S. warned Saudis of threats in Congress
Generated Headline Text: Romanian anti-corruption prosecutors open investigation into Liviu Dragnea on suspicion of forming criminal group to siphon off cash from state projects
Actual Headline Text   : Romanian ruling party leader investigated over 'criminal group'
Generated Headline Text: environmental activist endorsed Hillary Clinton for U.S. president
Actual Headline T